# EXEMPLO 2 — HyDE: Hypothetical Document Embeddings (Mínimo)
## Aula 6 · MBA RAG & CAG Aplicados a Direito e Segurança Pública

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

**Objetivo:** Demonstrar HyDE em 4 células, mostrando o antes e depois do gap semântico.

**Tempo:** 15 minutos

## Instalação

In [ ]:
!pip install sentence-transformers langchain-openai faiss-cpu --quiet
print('OK')

## Imports e Corpus

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import warnings
warnings.filterwarnings('ignore')

# Corpus juridico simples
corpus = [
    'A busca pessoal sem mandado judicial requer fundada suspeita objetiva e concreta conforme art 240 paragrafo 2 CPP. A Jurisprudencia do STJ exige elemento concreto, nao mera intuicao policial.',
    'A cadeia de custodia nos termos do art 158-A do CPP e o conjunto de procedimentos para documentar a historia cronologica do vestigio desde seu reconhecimento ate o descarte.',
    'O STF no HC 143641 determinou substituicao de prisao preventiva por domiciliar para gestantes e maes de criancas ate 12 anos nos termos do art 318 incisos IV e V do CPP.',
    'O compartilhamento de relatorios do COAF com o MP sem autorizacao judicial e constitucional conforme tese do RE 1055941 Tema 990 do STF.',
    'O reconhecimento facial com vies algoritmico nao pode ser usado como prova direta em processo penal servindo apenas como elemento investigativo sujeito a confirmacao humana.',
]

# Encoder
try:
    encoder = SentenceTransformer('BAAI/bge-m3')
    print('BGE-M3 carregado')
except:
    encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    print('Fallback: MiniLM')

# Indexar corpus no FAISS
embs = encoder.encode(corpus, normalize_embeddings=True)
dim = embs.shape[1]
idx = faiss.IndexFlatIP(dim)
idx.add(embs.astype('float32'))
print(f'FAISS indexado: {idx.ntotal} vetores de {dim}d')

## Pipeline HyDE vs Busca Direta

In [ ]:
# LLM para gerar documento hipotetico
LLM_ON = False
try:
    llm = ChatOpenAI(
        model='meta-llama/Llama-3.1-8B-Instruct',
        base_url='http://localhost:8000/v1',
        api_key='dummy',
        temperature=0.3,
        max_tokens=200,
    )
    llm.invoke('ok')
    LLM_ON = True
    print('vLLM conectado')
except:
    print('vLLM offline - usando hipoteticos pre-gerados')

# Hipoteticos pre-gerados (fallback)
hipotetivos_pregeados = {
    'policial pode revistar alguem na rua?':
        'A realizacao de busca pessoal sem mandado judicial e condicionada, nos termos do art 240 paragrafo 2 do CPP, a presenca de fundada suspeita objetiva de que o indivíduo oculte arma proibida ou objeto ilicito.',
    'mae presa pode ficar em casa?':
        'Consoante HC 143641 do STF, e possivel a substituicao da prisao preventiva pela domiciliar para mulheres gestantes ou maes de criancas ate 12 anos, conforme art 318 incisos IV e V do CPP.',
}

HYDE_PROMPT = ChatPromptTemplate.from_messages([
    ('system', 'Escreva um trecho de 80-120 palavras de um documento juridico brasileiro que responderia a pergunta. Use linguagem tecnica formal.'),
    ('human', 'Pergunta: {question}\nDocumento hipotetico:'),
])

def hyde_search(query, top_k=3):
    # Busca direta
    q_emb = encoder.encode([query], normalize_embeddings=True)[0]
    D_direct, I_direct = idx.search(q_emb.reshape(1,-1).astype('float32'), top_k)

    # Gerar hipotetico
    if LLM_ON:
        chain = HYDE_PROMPT | llm | StrOutputParser()
        hipotetivo = chain.invoke({'question': query})
    else:
        hipotetivo = hipotetivos_pregeados.get(query.lower(), query + ' contexto juridico brasileiro normativa legal processual penal')

    # Busca com hipotetico
    h_emb = encoder.encode([hipotetivo], normalize_embeddings=True)[0]
    D_hyde, I_hyde = idx.search(h_emb.reshape(1,-1).astype('float32'), top_k)

    return {
        'query': query,
        'hipotetico': hipotetivo[:150],
        'direto': [(D_direct[0][i], corpus[I_direct[0][i]][:80]) for i in range(top_k)],
        'hyde': [(D_hyde[0][i], corpus[I_hyde[0][i]][:80]) for i in range(top_k)],
    }

# Testar
for q in ['policial pode revistar alguem na rua?', 'mae presa pode ficar em casa?']:
    r = hyde_search(q)
    print(f'Query: {r["query"]}')
    print(f'  Hipotetico: {r["hipotetico"]}...')
    print(f'  Direto top-1: score={r["direto"][0][0]:.3f} | {r["direto"][0][1]}...')
    print(f'  HyDE top-1:  score={r["hyde"][0][0]:.3f} | {r["hyde"][0][1]}...')
    delta = r['hyde'][0][0] - r['direto'][0][0]
    print(f'  Delta score: {delta:+.3f}')
    print()

## Referências (ABNT)

GAO, L. et al. **Precise Zero-Shot Dense Retrieval without Relevance Labels**. In: *ACL*, Toronto, 2023. Disponível em: <https://arxiv.org/abs/2212.10496>.

---
*MBA em RAG & CAG — Aula 6, Exemplo 2*